# PyTorch ResNet18 Example
This notebook demonstrates loading a pretrained ResNet18 model and preparing random data for inference.

In [5]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

In [4]:
# If you get import errors, run this cell:
%pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 15.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] [torchvision]
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Load pretrained ResNet18 model
model = resnet18(weights=ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/julih/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:02<00:00, 15.8MB/s]



In [7]:
# Create random input data and labels
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

### What does `data = torch.rand(1, 3, 64, 64)` mean?
- The shape `(1, 3, 64, 64)` represents a batch of images:
  - `1`: batch size (number of images in the batch)
  - `3`: number of channels (e.g., RGB color channels)
  - `64`: image height (pixels)
  - `64`: image width (pixels)
- This creates a single random image with 3 color channels and 64x64 pixels, suitable as input for models like ResNet18.

## Forward pass and parameter inspection

**Q: Run a forward pass through the model with the data and save it as preds. What should the shape of preds be? Verify your guess.**

In [8]:
# Forward pass
preds = model(data)
print('preds.shape:', preds.shape)  # Should be [1, 1000]

preds.shape: torch.Size([1, 1000])


### Why should preds.shape be [1, 1000]?
- The input batch has shape (1, 3, 64, 64): 1 image, 3 channels, 64x64 pixels.
- ResNet18 outputs a vector of length 1000 for each image (one score per ImageNet class).
- So, for a batch of 1 image, the output shape is (1, 1000):
  - 1: batch size (number of images)
  - 1000: number of output classes (ImageNet)

### Inspecting the first convolutional layer weights

In [9]:
w = model.conv1.weight
print('w:', w)

w: Parameter containing:
tensor([[[[-1.0419e-02, -6.1356e-03, -1.8098e-03,  ...,  5.6615e-02,
            1.7083e-02, -1.2694e-02],
          [ 1.1083e-02,  9.5276e-03, -1.0993e-01,  ..., -2.7124e-01,
           -1.2907e-01,  3.7424e-03],
          [-6.9434e-03,  5.9089e-02,  2.9548e-01,  ...,  5.1972e-01,
            2.5632e-01,  6.3573e-02],
          ...,
          [-2.7535e-02,  1.6045e-02,  7.2595e-02,  ..., -3.3285e-01,
           -4.2058e-01, -2.5781e-01],
          [ 3.0613e-02,  4.0960e-02,  6.2850e-02,  ...,  4.1384e-01,
            3.9359e-01,  1.6606e-01],
          [-1.3736e-02, -3.6746e-03, -2.4084e-02,  ..., -1.5070e-01,
           -8.2230e-02, -5.7828e-03]],

         [[-1.1397e-02, -2.6619e-02, -3.4641e-02,  ...,  3.2521e-02,
            6.6221e-04, -2.5743e-02],
          [ 4.5687e-02,  3.3603e-02, -1.0453e-01,  ..., -3.1253e-01,
           -1.6051e-01, -1.2826e-03],
          [-8.3730e-04,  9.8420e-02,  4.0210e-01,  ...,  7.0789e-01,
            3.6887e-01,  1.2455e-

In [10]:
print('w.grad before backward:', w.grad)  # Should be None

w.grad before backward: None


### Compute loss and inspect grad_fn

**Q: Create a CrossEntropy loss object, and use it to compute a loss using ‘labels’ and ‘preds’, saved as ‘loss’. Print loss because we will need it for later. Print the last mathematical operation that created ‘loss’.**

In [11]:
ce = torch.nn.CrossEntropyLoss()
loss = ce(preds, labels)
print('loss:', loss)
print('loss.grad_fn:', loss.grad_fn)

loss: tensor(3566.9321, grad_fn=<DivBackward1>)
loss.grad_fn: <DivBackward1 object at 0x117647700>


In [12]:
loss.backward()

### What does `backward()` do in PyTorch?

- `backward()` computes **gradients** of a scalar output (like the loss) with respect to all tensors in the computation graph that have `requires_grad=True`.
- It performs **reverse-mode automatic differentiation** (backpropagation), traversing the computation graph from the output back to the inputs.
- After calling `backward()`, the `.grad` attribute of each leaf tensor (e.g., model parameters) contains the gradient of the output with respect to that tensor.
- These gradients are then used by optimizers (like SGD) to update the model parameters and minimize the loss during training.

In [13]:
print('w after backward:', w)  # Should be unchanged
print('w.grad after backward:', w.grad)  # Should now be a tensor

w after backward: Parameter containing:
tensor([[[[-1.0419e-02, -6.1356e-03, -1.8098e-03,  ...,  5.6615e-02,
            1.7083e-02, -1.2694e-02],
          [ 1.1083e-02,  9.5276e-03, -1.0993e-01,  ..., -2.7124e-01,
           -1.2907e-01,  3.7424e-03],
          [-6.9434e-03,  5.9089e-02,  2.9548e-01,  ...,  5.1972e-01,
            2.5632e-01,  6.3573e-02],
          ...,
          [-2.7535e-02,  1.6045e-02,  7.2595e-02,  ..., -3.3285e-01,
           -4.2058e-01, -2.5781e-01],
          [ 3.0613e-02,  4.0960e-02,  6.2850e-02,  ...,  4.1384e-01,
            3.9359e-01,  1.6606e-01],
          [-1.3736e-02, -3.6746e-03, -2.4084e-02,  ..., -1.5070e-01,
           -8.2230e-02, -5.7828e-03]],

         [[-1.1397e-02, -2.6619e-02, -3.4641e-02,  ...,  3.2521e-02,
            6.6221e-04, -2.5743e-02],
          [ 4.5687e-02,  3.3603e-02, -1.0453e-01,  ..., -3.1253e-01,
           -1.6051e-01, -1.2826e-03],
          [-8.3730e-04,  9.8420e-02,  4.0210e-01,  ...,  7.0789e-01,
            3.6887

### Why is `w` unchanged but `w.grad` changed after backward?
- `w` is the actual weight tensor of the model. The backward pass only computes gradients; it does not update weights.
- `w.grad` is the gradient tensor, populated by the backward pass. It stores how much each weight should change to reduce the loss.
- The optimizer step (e.g., `sgd.step()`) uses `w.grad` to update `w`.
- So after `loss.backward()`, `w` is unchanged, but `w.grad` contains the computed gradients.

In [14]:
print('loss.grad:', loss.grad)  # Should be None (not a leaf node)
print('loss.requires_grad:', loss.requires_grad)  # Should be True
print('labels.requires_grad:', labels.requires_grad)  # Should be False

loss.grad: None
loss.requires_grad: True
labels.requires_grad: False


/var/folders/ft/lc32cctj0bvc8pm8hmc3_9q80000gn/T/ipykernel_51920/979939321.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:497.)
  print('loss.grad:', loss.grad)  # Should be None (not a leaf node)


### Why are these values as shown?

- `loss.grad` is `None` because `loss` is not a leaf tensor. Only leaf tensors have their `.grad` populated by default. To get the gradient for non-leaf tensors, you must call `.retain_grad()` on them before backward.
- `loss.requires_grad` is `True` because `loss` is computed from tensors that require gradients (the model parameters), so it participates in the computation graph.
- `labels.requires_grad` is `False` because `labels` are target values, not parameters to optimize, so gradients are not needed for them.

In [15]:
try:
    loss.backward()
except RuntimeError as e:
    print('Second backward error:', e)

Second backward error: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.


### Why is calling `backward()` a second time an error?

- By default, PyTorch **frees the computation graph** after you call `backward()` to save memory, since the graph is no longer needed.
- If you try to call `backward()` again on the same graph (without specifying `retain_graph=True`), PyTorch raises an error because the graph has already been deleted.
- If you need to call `backward()` multiple times on the same graph (e.g., for multiple backward passes), you must pass `retain_graph=True` to the first `backward()` call: `loss.backward(retain_graph=True)`.
- In most training scenarios, you only need to call `backward()` once per forward pass, so the default behavior is efficient and safe.

backward() computes gradients for all tensors with requires_grad=True by performing backpropagation, and that these gradients are used by optimizers to update model parameters

In [16]:
sgd = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
sgd.step()
print('w after optimizer step:', w)

w after optimizer step: Parameter containing:
tensor([[[[ 3.6881e-01, -3.0100e-01,  9.2419e-02,  ...,  2.7278e-01,
           -5.5597e-01,  5.6106e-01],
          [-4.7982e-02,  3.6499e-01, -1.1694e-01,  ..., -7.2124e-01,
            1.8282e-01, -1.0316e-01],
          [-2.4233e-01, -5.0546e-02,  7.9265e-02,  ...,  7.0480e-01,
           -2.7206e-02,  1.7604e-01],
          ...,
          [ 3.8833e-01, -6.0307e-01, -5.7990e-02,  ..., -1.1093e+00,
           -3.5813e-01, -3.1528e-01],
          [ 1.1622e-01, -6.0597e-03, -1.3935e-01,  ...,  4.1006e-01,
            5.4260e-01,  7.3546e-03],
          [-9.6070e-02, -2.0764e-01, -5.8767e-01,  ..., -6.7333e-01,
           -8.7917e-02,  8.7004e-02]],

         [[ 3.7944e-01, -1.0984e-01, -1.7838e-01,  ..., -3.3055e-01,
           -1.9802e-01,  2.6226e-01],
          [ 3.0200e-01,  4.3423e-01, -5.5982e-01,  ...,  1.7422e-02,
           -4.0201e-01,  5.3183e-01],
          [ 2.1314e-01,  2.8206e-01,  8.9079e-02,  ...,  6.6466e-01,
            

In [17]:
print('loss after optimizer step:', loss)

loss after optimizer step: tensor(3566.9321, grad_fn=<DivBackward1>)


In [18]:
model.zero_grad()
print('w.grad after zero_grad:', w.grad)  # Should be zero

w.grad after zero_grad: None


### What does `torch.optim.SGD` mean? What is an optimizer step?

- `torch.optim.SGD` stands for **Stochastic Gradient Descent**, a common optimization algorithm used to update model parameters (like weights) using their gradients.
- You create an optimizer (e.g., `sgd = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)`) and pass it the model's parameters and hyperparameters like learning rate (`lr`) and momentum.
- An **optimizer step** (`sgd.step()`) updates each parameter by subtracting a fraction of its gradient (scaled by the learning rate), moving the parameters in the direction that reduces the loss.

### Why is `w.grad` zero after `model.zero_grad()`?

- After calling `backward()`, each parameter's `.grad` attribute contains the computed gradient.
- If you call `sgd.step()` without clearing these gradients, the next backward pass will accumulate (add to) the existing gradients, which is usually not what you want.
- `model.zero_grad()` sets all parameter gradients to zero, so the next backward pass starts fresh.
- Therefore, after `model.zero_grad()`, `w.grad` is zero (or a tensor of zeros), ready for the next iteration.

### Why is it bad if the next backward pass accumulates gradients?

- If you do not zero out the gradients before the next backward pass, PyTorch will **add the new gradients to the existing ones** in each parameter's `.grad` attribute.
- This means the gradients will be **incorrect** for the current batch—they will include contributions from previous batches, leading to wrong parameter updates.
- As a result, the optimizer will make updates based on the sum of multiple gradients, not just the current one, which can cause the model to train incorrectly or not converge.
- **Best practice:** Always call `model.zero_grad()` (or `optimizer.zero_grad()`) before each backward pass to ensure gradients are only from the current batch.

### What does `momentum=0.9` mean in SGD?

- **Momentum** is a technique to help accelerate SGD in the relevant direction and dampen oscillations.
- When you set `momentum=0.9`, each parameter update is influenced not only by the current gradient, but also by the previous update (the "velocity").
- This means the optimizer remembers the direction it was moving in the past and continues in that direction, smoothing out updates and helping escape shallow local minima.
- A value like `0.9` means each update is 90% influenced by the previous update and 10% by the current gradient.
- In practice, momentum often leads to faster convergence and more stable training.

### Q: What should the `grad` attribute for `w` be after `zero_grad()`? Verify.

A: It should be zero. After calling `model.zero_grad()`, all parameter gradients are reset to zero, so `w.grad` will be a tensor of zeros.

---

### Q: Determine, without running, whether the following code will successfully execute:
```python
data1 = torch.zeros(1, 3, 64, 64)
data2 = torch.ones(1, 3, 64, 64)

predictions1 = model(data1)
predictions2 = model(data2)

l = torch.nn.CrossEntropyLoss()
loss1 = l(predictions1, labels)
loss2 = l(predictions2, labels)

loss1.backward()
loss2.backward()
```
A: **Yes!** `loss2.backward()` will work because it uses a different computation graph (from `predictions2`). The intermediate values for `loss1` are freed after its backward, but `loss2` has its own graph.

#### Explanation:
Each forward pass creates a new computation graph. Since `predictions2` is computed from new data, its backward pass is independent of the first and will succeed.

---

### Q: As above, will the following code execute?
```python
data1 = torch.zeros(1, 3, 64, 64)
data2 = torch.ones(1, 3, 64, 64)

predictions1 = model(data1)
predictions2 = model(data1)

l = torch.nn.CrossEntropyLoss()
loss1 = l(predictions1, labels)
loss2 = l(predictions2, labels)

loss1.backward()
loss2.backward()
```
A: **Yes!** Again, `loss2.backward()` will work because `predictions2` is a new forward pass, so it has its own computation graph.

#### Explanation:
Even though both predictions use the same data, each call to `model(data1)` creates a new computation graph, so both backward passes are independent.

---

### Q: As above, will the following code execute?
```python
data1 = torch.zeros(1, 3, 64, 64)
data2 = torch.ones(1, 3, 64, 64)

predictions1 = model(data1)
predictions2 = model(data2)

l = torch.nn.CrossEntropyLoss()
loss1 = l(predictions1, labels)
loss2 = l(predictions1, labels)

loss1.backward()
loss2.backward()
```
A: **No!** `loss2.backward()` will fail because it tries to use the same computation graph as `loss1`, which has already been freed after the first backward.

#### Explanation:
Both losses are computed from `predictions1`, so after the first backward, the graph is gone. The second backward will raise an error unless you use `retain_graph=True` in the first backward.

---

### Q: For the ones that don’t execute, how can you modify the code to make it work?
A: Change the first `.backward()` call to use `retain_graph=True`:
```python
loss1.backward(retain_graph=True)
loss2.backward()
```
This keeps the computation graph after the first backward so it can be used again.

---

### Q: What will be the output of the following code?
```python
predictions1 = model(data)
l = torch.nn.CrossEntropyLoss()
loss1 = l(predictions1, labels)
loss1.backward(retain_graph=True)

w = model.conv1.weight.grad[0][0][0][0]
a = w.item()

loss1.backward()
b = w.item()

model.zero_grad()
c = w.item()

print(b//a, c)
```
A: `2.0, 0.0`

#### Explanation:
- The first backward computes the gradient and stores it in `w.grad` (let's call this value `a`).
- The second backward (with the same loss and `retain_graph=True` used before) adds the gradient again, so `w.grad` doubles (now `b = 2*a`).
- After `zero_grad()`, `w.grad` is reset to zero (`c = 0`).
- Integer division `b//a` gives `2.0`.

---

### Q: What will be the output of the following code?
```python
predictions1 = model(data)
l = torch.nn.CrossEntropyLoss()
loss1 = l(predictions1, labels)
loss1.backward(retain_graph=True)

a = model.conv1.weight.grad[0][0][0][0]

loss1.backward()
b = model.conv1.weight.grad[0][0][0][0]

model.zero_grad()
c = model.conv1.weight.grad[0][0][0][0]

print(b//a, c)
```
A: `tensor(nan) tensor(0.)`

#### Explanation:
- After the first backward, `a` is the gradient value.
- The second backward adds the gradient again, but since `a` might be zero, dividing by zero can result in `nan` (not a number).
- After `zero_grad()`, the gradient is zero.
- So the output is `tensor(nan) tensor(0.)`.

### Why could `a` be zero in the second example, but not in the first?

- In the **first scenario**, `a` is the gradient value after the first backward pass, and `b` is double that value. If `a` is nonzero, `b//a` is 2.0. If `a` is zero, `b` is also zero, so `b//a` would be undefined (nan), but the question assumes `a` is nonzero.
- In the **second scenario**, you explicitly check `a` after the first backward, but if the computed gradient happens to be zero (for example, due to the specific input or model weights), then dividing by zero in `b//a` will produce `nan`.
- The difference is: in the first scenario, the question assumes `a` is nonzero (so integer division works), but in the second, it's possible for `a` to be zero, depending on the model and data.
- In practice, gradients can be zero if the loss surface is flat at that point, or the input doesn't affect the parameter being checked.

#### Summary:
- If `a` is zero, `b//a` is `nan`.
- If `a` is nonzero, `b//a` is 2.0.
- Whether `a` is zero depends on the model, data, and which parameter's gradient you inspect.

### Clearer difference between the 1st and 2nd scenarios

#### 1st Scenario:
```python
w = model.conv1.weight.grad[0][0][0][0]
a = w.item() //makes a nonzero
loss1.backward()
b = w.item()
model.zero_grad()
c = w.item()
print(b//a, c)
```
- Here, `a` is the gradient value after the first backward pass.
- The question assumes `a` is nonzero, so `b//a` is 2.0 (because the second backward doubles the gradient).
- If `a` were zero, `b` would also be zero, and `b//a` would be undefined (nan), but the scenario is constructed so `a` is not zero.
- In practice, with typical data and model weights, gradients are rarely exactly zero unless the loss surface is flat or the input doesn't affect the parameter.

#### 2nd Scenario:
```python
a = model.conv1.weight.grad[0][0][0][0]
loss1.backward()
b = model.conv1.weight.grad[0][0][0][0]
model.zero_grad()
c = model.conv1.weight.grad[0][0][0][0]
print(b//a, c)
```
- Here, you explicitly check `a` after the first backward, but you don't assume it's nonzero.
- If `a` is zero (possible if the gradient for that parameter is zero), then `b//a` is `nan`.
- This scenario allows for the possibility that the gradient is zero, depending on the model, data, and which parameter you inspect.

#### Why can't `a` be zero in the 1st scenario?
- The 1st scenario is written with the assumption that `a` is nonzero, so the division makes sense and gives 2.0.
- If `a` were zero, the division would be undefined, but the question is posed to illustrate gradient accumulation (doubling), not division by zero.
- In most practical cases, gradients are not exactly zero unless the model is at a stationary point or the input doesn't affect the parameter.

#### Summary:
- The 1st scenario assumes `a` is nonzero to demonstrate doubling.
- The 2nd scenario allows for `a` to be zero, so division by zero can occur.
- Whether `a` is zero depends on the model, data, and parameter.

### Does `a = w.item()` guarantee a nonzero value?

- No, `a = w.item()` does **not** guarantee that `a` is nonzero.
- `w.item()` simply extracts the value of the selected element from the gradient tensor.
- Whether `a` is zero or nonzero depends on the model weights, input data, and the loss function.
- In most practical cases, gradients are nonzero, but it's possible for a specific parameter's gradient to be zero (e.g., if the loss surface is flat at that point or the input doesn't affect that weight).
- The scenario assumes `a` is nonzero for illustration, but you should not assume this in general code.

### Why does the 1st question assume `a` is nonzero?

- The 1st question is designed to illustrate how gradients accumulate when you call `backward()` multiple times.
- It assumes `a` is nonzero so the division `b//a` produces a meaningful result (2.0), showing that the gradient doubles after two backward passes.
- If `a` were zero, the division would be undefined (nan), which would not demonstrate the intended concept.
- In real code, you should always check for zero before dividing, but for teaching purposes, the question assumes a typical case where gradients are nonzero.

### Q: What is wrong with the following code?
```python
learning_rate = 0.01
for f in net.parameters():
    f.data.sub(f.grad.data * learning_rate)
```
A: The `sub` call should be `sub_`, which will correctly perform the expected in-place operation.

#### Explanation:
- `sub` returns a new tensor, but does not modify `f.data` in-place.
- `sub_` is the in-place version, updating the parameter directly.
- Using `sub_` ensures the weights are updated as intended during training.

---

### Q: Order the following steps of the training loop correctly (there are multiple correct answers, but one typical setup):
- optimizer.step(), optimizer.zero_grad(), loss.backward(), output = net(input), loss = criterion(output, target)

A: One typical order is:
```python
optimizer.zero_grad()
output = net(input)
loss = criterion(output, target)
loss.backward()
optimizer.step()
```

#### Explanation:
- Zero gradients first to avoid accumulation.
- Forward pass to get output.
- Compute loss.
- Backward pass to compute gradients.
- Step optimizer to update parameters.

---

### Q: What will be the output of the following code?
```python
net = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
target = torch.rand(1, 1000)
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
criterion = torch.nn.CrossEntropyLoss()
orig = net.conv1.weight.clone()[0, 0, 0, 0]
weight = net.conv1.weight[0, 0, 0, 0]
# 1
optimizer.zero_grad()
print(f"{weight == orig}")
# 2
output = net(data)
loss = criterion(output, target)
print(f"{weight == orig}")
# 3
loss.backward()
print(f"{weight == orig}")
# 4
optimizer.step()
print(f"{weight == orig}")
```
A:
```
True
True
True
False
```

#### Explanation:
- Before optimizer step, weights are unchanged.
- After optimizer step, weights are updated, so comparison is False.

---

### Q: Implement a neural network with one hidden layer for grayscale 32x32 images, using nn.Linear, F.relu, torch.flatten.

A:
```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(32 * 32, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
```

#### Explanation:
- Input is flattened.
- First linear layer maps to 100 features, then ReLU.
- Second linear layer maps to 10 classes.

---

### Q: Using two lines of code, verify a forward pass through the above network.
```python
net = Net()
preds = net.forward(torch.randn(1, 1, 32, 32))
```

#### Explanation:
- Instantiates the network.
- Runs a random input through the forward pass.

---

### Q: Without running, what would the following statement yield?
```python
net = Net()
print(len(list(net.parameters())))
```
A: 4

#### Explanation:
- Two layers, each with weight and bias.

---

### Q: Get the names of the net parameters
```python
print([name for name, _ in net.named_parameters()])
```
A: ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']

#### Explanation:
- Each layer has a weight and bias, named accordingly.

---

### Q: What network layer is the following statement referring to? What will it evaluate to?
```python
print(list(net.parameters())[1].size())
```
A: fc1.bias. torch.Size([100])

#### Explanation:
- Second parameter is the bias of the first layer, which has 100 units.

---

### Q: Implement a neural network based on the schematic, using nn.Conv2d, nn.Linear, F.max_pool2d, F.relu, torch.flatten.

A:
```python
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.max_pool2d(x, 2)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.max_pool2d(x, 2)
        x = F.relu(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x
```

#### Explanation:
- Two conv layers, two max pools, three fully connected layers, ReLU after each pool and FC.

---

### Q: Modify the above code to use nn.MaxPool2d instead of F.max_pool2d

A:
```python
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.maxpool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = F.relu(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x
```

#### Explanation:
- Uses nn.MaxPool2d module for pooling instead of functional API.

---

### Q: Try increasing the width of your network by increasing the number of output channels of the first convolution from 6 to 12. What else do you need to change?

A:
```python
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 12, 5)
        self.conv2 = nn.Conv2d(12, 16, 5) # input channels updated
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.maxpool = nn.MaxPool2d(2, 2)
```

#### Explanation:
- When increasing output channels in conv1, update input channels in conv2 to match.

---

### Q: The following dataset loading code runs but are there mistakes? What are the implications and fixes?
```python
trainset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=False, num_workers=2)
testset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)
```
A: Two mistakes:
- Not shuffling the train data loader.
- Loading train data in testset and test data in trainset.

#### Fixes:
- Set `shuffle=True` for trainloader.
- Use `train=True` for trainset and `train=False` for testset.

---

### Q: Write 2 lines of code to get random training images from the dataloader (assuming errors above are fixed).
```python
dataiter = iter(trainloader)
images, labels = next(dataiter)
```

#### Explanation:
- Creates an iterator and gets a batch of images and labels.

---

### Q: The following training code runs but are there mistakes (computational inefficiencies)? What are the implications and fixes?
```python
running_loss = 0.0
for i, data in enumerate(trainloader, 0):
    inputs, labels = data
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss
    if i % 2000 == 1999:
        print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
        running_loss = 0.0
        break
```
A: Two mistakes:
- Missing `optimizer.zero_grad()`; gradients will accumulate.
- `running_loss` should use `loss.item()` to avoid memory issues.

#### Fixes:
- Add `optimizer.zero_grad()` before forward pass.
- Use `running_loss += loss.item()`.

---

### Q: The following evaluation code runs but are there mistakes (computational inefficiencies)? What are the implications and fixes?
```python
correct = 0
total = 0
for data in testloader:
    images, labels = data
    outputs = net(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum()
print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')
```
A: Two mistakes:
- Should use `torch.no_grad()` to disable autograd for evaluation.
- Missing `.item()` after sum; otherwise, tensors stay in the computation graph.

#### Fixes:
- Wrap loop in `with torch.no_grad():`.
- Use `correct += (predicted == labels).sum().item()`.


### Q: What will be the output of the following code?
```python
net = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
target = torch.rand(1, 1000)
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
criterion = torch.nn.CrossEntropyLoss()
orig = net.conv1.weight.clone()[0, 0, 0, 0]
weight = net.conv1.weight[0, 0, 0, 0]
# 1
optimizer.zero_grad()
print(f"{weight == orig}")
# 2
output = net(data)
loss = criterion(output, target)
print(f"{weight == orig}")
# 3
loss.backward()
print(f"{weight == orig}")
# 4
optimizer.step()
print(f"{weight == orig}")
```
A:
```
True
True
True
False
```
#### Explanation:
- Before optimizer step, weights are unchanged.
- After optimizer step, weights are updated, so comparison is False.

---

### Q: Implement a neural network with one hidden layer for grayscale 32x32 images. Use nn.Linear, F.relu, torch.flatten.

A:
```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(32 * 32, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
```
#### Explanation:
- Input is flattened.
- First linear layer maps to 100 features, then ReLU.
- Second linear layer maps to 10 classes.
- x = F.relu(self.fc1(x)) applies a linear transformation and ReLU non-linearity to extract features.
x = self.fc2(x) maps those features to the output classes for prediction.
Linear layers learn weighted combinations of inputs; the 100 is the number of hidden features.
Nonlinear layers (ReLU) allow the network to learn complex, non-linear relationships.
fc1 extracts features; fc2 maps those features to output classes.
Both are needed for richer, more powerful models
torch.flatten(x, 1) flattens all dimensions except the batch, so each sample becomes a vector.
F.relu(self.fc1(x)): F is the functional API; ReLU is applied after the linear layer to introduce non-linearity.
self.fc2(x) maps the features to the output classes
---

### Q: Using two lines of code, verify a forward pass through the above network.
```python
net = Net()
preds = net.forward(torch.randn(1, 1, 32, 32))
```
#### Explanation:
- Instantiates the network.
- Runs a random input through the forward pass.

---

### Q: Without running, what would the following statement yield?
```python
net = Net()
print(len(list(net.parameters())))
```
A: 4
#### Explanation:
- Two layers, each with weight and bias.

---

### Q: Get the names of the net parameters
```python
print([name for name, _ in net.named_parameters()])
```
A: ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']
#### Explanation:
- Each layer has a weight and bias, named accordingly.

---

### Q: What network layer is the following statement referring to? What will it evaluate to?
```python
print(list(net.parameters())[1].size())
```
A: fc1.bias. torch.Size([100])
#### Explanation:
- Second parameter is the bias of the first layer, which has 100 units.
A markdown cell has been added explaining that list(net.parameters())[1].size() refers to fc1.bias, and the size is 100 because the first fully connected layer outputs 100 features, each with its own bias. Let me know if you want more details or further examples!
---

### Q: Implement a neural network based on the schematic, using nn.Conv2d, nn.Linear, F.max_pool2d, F.relu, torch.flatten.

A:
```python
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.max_pool2d(x, 2)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.max_pool2d(x, 2)
        x = F.relu(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x
```
#### Explanation:
- Two conv layers, two max pools, three fully connected layers, ReLU after each pool and FC.

---

### Q: Modify the above code to use nn.MaxPool2d instead of F.max_pool2d
A:
```python
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.maxpool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = F.relu(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x
```
#### Explanation:
- Uses nn.MaxPool2d module for pooling instead of functional API.

---

### Q: Try increasing the width of your network by increasing the number of output channels of the first convolution from 6 to 12. What else do you need to change?
A:
```python
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 12, 5)
        self.conv2 = nn.Conv2d(12, 16, 5) # input channels updated
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.maxpool = nn.MaxPool2d(2, 2)
```
#### Explanation:
- When increasing output channels in conv1, update input channels in conv2 to match.

---

### Q: The following dataset loading code runs but are there mistakes? What are the implications and fixes?
```python
trainset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=False, num_workers=2)
testset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)
```
A: Two mistakes:
- Not shuffling the train data loader.
- Loading train data in testset and test data in trainset.
#### Fixes:
- Set `shuffle=True` for trainloader.
- Use `train=True` for trainset and `train=False` for testset.

---

### Q: Write 2 lines of code to get random training images from the dataloader (assuming errors above are fixed).
```python
dataiter = iter(trainloader)
images, labels = next(dataiter)
```
#### Explanation:
- Creates an iterator and gets a batch of images and labels.

---

### Q: The following training code runs but are there mistakes (computational inefficiencies)? What are the implications and fixes?
```python
running_loss = 0.0
for i, data in enumerate(trainloader, 0):
    inputs, labels = data
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss
    if i % 2000 == 1999:
        print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
        running_loss = 0.0
        break
```
A: Two mistakes:
- Missing `optimizer.zero_grad()`; gradients will accumulate.
- `running_loss` should use `loss.item()` to avoid memory issues.
#### Fixes:
- Add `optimizer.zero_grad()` before forward pass.
- Use `running_loss += loss.item()`.

---

### Q: The following evaluation code runs but are there mistakes (computational inefficiencies)? What are the implications and fixes?
```python
correct = 0
total = 0
for data in testloader:
    images, labels = data
    outputs = net(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum()
print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')
```
A: Two mistakes:
- Should use `torch.no_grad()` to disable autograd for evaluation.
- Missing `.item()` after sum; otherwise, tensors stay in the computation graph.
#### Fixes:
- Wrap loop in `with torch.no_grad():`.
- Use `correct += (predicted == labels).sum().item()`.

---

### PyTorch Cheatsheet
It may take you some time to feel comfortable with PyTorch, and that’s okay! PyTorch is a powerful tool for deep learning development. After the above exercises, you can go through the Quickstart tutorial here, which will cover more aspects including Save & Load Model, and Datasets and Dataloaders. As you learn the API, you may find it useful to remember key usage patterns; a PyTorch cheatsheet I like is found here: https://pytorch.org/tutorials/beginner/quickstart_tutorial.html

And that covers our fundamentals of PyTorch! Congratulations – you’re now equipped to start tackling more complex deep learning code that leverages PyTorch.

### What is the purpose of these lines?
```python
x = F.relu(self.fc1(x))
x = self.fc2(x)
```
- `x = F.relu(self.fc1(x))` applies a linear transformation (fully connected layer) to the input, then applies the ReLU activation function.
    - The linear layer (`fc1`) projects the input to a new feature space (here, 100 features).
    - ReLU introduces non-linearity, allowing the network to learn more complex functions.
- `x = self.fc2(x)` applies another linear transformation, mapping the features to the output classes (here, 10 classes).
    - This layer produces the final scores for each class.

#### Summary:
- The first line extracts features and applies non-linearity.
- The second line maps those features to the output classes for prediction.

### Why use both linear and nonlinear layers? What is the 100 for? Why need `fc1` and `fc2`?

- **Linear layer (`fc1`)**: Projects the input into a new feature space. The number `100` is the number of features (hidden units) in this space. It allows the network to learn useful intermediate representations.
- **Nonlinear layer (ReLU)**: Adds non-linearity, enabling the network to learn complex, non-linear relationships. Without non-linearity, stacking linear layers would just be equivalent to a single linear transformation.
- **Why need `fc1` and `fc2`?**
    - `fc1` extracts features from the input and transforms them into a higher-dimensional space (100 features).
    - `fc2` maps those features to the output classes (10 classes for classification).
    - Having two layers (with non-linearity in between) allows the network to learn richer, more complex mappings from input to output.

#### Summary:
- Linear layers learn weighted combinations of inputs.
- Nonlinear layers (like ReLU) allow the network to model complex functions.
- The hidden size (100) controls the capacity of the network to learn intermediate features.
- `fc1` is for feature extraction; `fc2` is for output mapping.

### What does `torch.flatten(x, 1)` mean? Why `F.relu(self.fc1(x))`? Why `self.fc2(x)`?

- **`torch.flatten(x, 1)`**: Flattens the input tensor `x` starting from dimension 1 (the second dimension). This keeps the batch dimension (dimension 0) intact, so each image in the batch is flattened into a vector, but the batch structure is preserved. For example, shape (batch, channels, height, width) becomes (batch, features).

- **Why `F.relu(self.fc1(x))`?**
    - `F` is the `torch.nn.functional` module, which provides stateless functions like `relu`.
    - `self.fc1(x)` applies the first linear (fully connected) layer to `x`.
    - `F.relu(...)` applies the ReLU activation function to the output of `fc1`. This introduces non-linearity, allowing the network to learn more complex patterns.
    - We apply ReLU right after `fc1` because activations after a linear layer are typically passed through a non-linearity to enable the network to approximate non-linear functions.

- **Why `self.fc2(x)`?**
    - After extracting features and applying non-linearity, we use `fc2` to map the features to the output classes.
    - `self.fc2(x)` produces the final output scores (logits) for each class.

#### Summary:
- Flatten with `1` keeps the batch dimension and flattens the rest.
- `F.relu(self.fc1(x))` applies a linear transformation and non-linearity to extract features.
- `self.fc2(x)` maps those features to the output classes for prediction.

### Why apply ReLU after `fc1(x)` and not after `fc2(x)` or just `x`?

- **ReLU after `fc1(x)`**: The purpose is to introduce non-linearity between the two linear layers. This allows the network to learn more complex, non-linear relationships.
    - If you put ReLU after `fc2(x)`, the output would be non-negative, which is not suitable for classification logits (which can be positive or negative and are typically passed to a loss function like CrossEntropyLoss).
    - If you put ReLU on the input `x` directly, you would lose information by zeroing out all negative values before any feature extraction happens.
- **Standard practice**: Apply non-linearity after hidden layers, but not after the final output layer in classification tasks, so the output can represent any real value (logits).

#### Summary:
- ReLU after `fc1(x)` enables the network to learn non-linear features.
- The final layer (`fc2`) should not have ReLU for classification, so the output logits can take any value.

### Why does `torch.randn(1, 1, 32, 32)` have 4 numbers? What do they mean?

- **Shape explanation:**
    - The input tensor shape is `(batch_size, channels, height, width)`.
    - `1, 1, 32, 32` means:
        - `1`: Batch size (number of images in the batch)
        - `1`: Number of channels (e.g., grayscale image)
        - `32`: Height (pixels)
        - `32`: Width (pixels)
- **Why 4 numbers?**
    - Convolutional neural networks expect 4D input: `[batch, channel, height, width]`.
    - Batch dimension allows processing multiple images at once.
    - Channel dimension distinguishes grayscale (1) vs. RGB (3).
    - Height and width are spatial dimensions.

#### Summary:
- `torch.randn(1, 1, 32, 32)` creates a batch of 1 grayscale image, 32x32 pixels.
- All 4 dimensions are required for CNN input.

### Why does `len(list(net.parameters()))` return 4? What are the parameter names?

- **Explanation:**
    - The network `Net()` has two linear layers: `fc1` and `fc2`.
    - Each linear layer has two parameters:
        - `weight`: The matrix of learnable weights
        - `bias`: The vector of learnable biases
    - So, for two layers, there are 4 parameters in total:
        - `fc1.weight`, `fc1.bias`, `fc2.weight`, `fc2.bias`
- **Code:**
    - `len(list(net.parameters()))` counts all parameter tensors (weights and biases).
    - `net.named_parameters()` returns their names and values.
    - `[name for name, _ in net.named_parameters()]` lists the parameter names.

#### Summary:
- 4 parameters: weights and biases for each of the two linear layers.
- Names: `fc1.weight`, `fc1.bias`, `fc2.weight`, `fc2.bias`.

### What does `list(net.parameters())[1].size()` refer to? Why is the bias size 100?

- **Which layer?**
    - `list(net.parameters())[1]` refers to the second parameter in the network, which is `fc1.bias` (the bias of the first fully connected layer).
- **What does it evaluate to?**
    - `fc1.bias` is a 1D tensor with shape `[100]`.
    - So, `print(list(net.parameters())[1].size())` outputs `torch.Size([100])`.
- **Why 100?**
    - The first linear layer `fc1` is defined as `nn.Linear(input_features, 100)`.
    - This means it outputs 100 features for each input.
    - Each output feature has its own bias term, so the bias vector has length 100.

#### Summary:
- The statement refers to the bias of the first fully connected layer (`fc1.bias`).
- The size is 100 because `fc1` outputs 100 features, each with its own bias.

### How are the layer sizes in the Net class determined?

- **conv1 = nn.Conv2d(1, 6, 5)**
  - Input channels: 1 (grayscale image)
  - Output channels: 6 (number of filters)
  - Kernel size: 5x5
- **conv2 = nn.Conv2d(6, 16, 5)**
  - Input channels: 6 (from previous layer)
  - Output channels: 16
  - Kernel size: 5x5
- **fc1 = nn.Linear(16 * 5 * 5, 120)**
  - After two conv layers and two 2x2 max pools, the feature map size is 5x5 with 16 channels:
    - Input image: 32x32
    - After conv1 (5x5 kernel, no padding): 28x28
    - After max pool (2x2): 14x14
    - After conv2 (5x5 kernel): 10x10
    - After max pool (2x2): 5x5
    - Channels: 16
    - Flattened size: 16 * 5 * 5 = 400
  - Output: 120 features
- **fc2 = nn.Linear(120, 84)**
  - Input: 120 (from previous layer)
  - Output: 84
- **fc3 = nn.Linear(84, 10)**
  - Input: 84
  - Output: 10 (number of classes)

#### Summary:
- The numbers are determined by the input image size, convolution/pooling operations, and the desired output classes.

### Why 6 output channels for conv1, and 16 for conv2?

- The number of output channels (filters) in each convolutional layer is a design choice made by the network creator.
- **conv1 = nn.Conv2d(1, 6, 5):**
  - Takes a grayscale image (1 channel) and outputs 6 feature maps.
  - Each of the 6 filters learns to detect different patterns in the input image.
- **conv2 = nn.Conv2d(6, 16, 5):**
  - Takes the 6 feature maps from conv1 as input and outputs 16 new feature maps.
  - Each of the 16 filters can learn more complex patterns by combining information from all 6 previous feature maps.
- The choice of 6 and 16 is inspired by the classic LeNet-5 architecture, which used these numbers for its convolutional layers. They are not required values, but are commonly used for small image classification networks.

**Summary:**
- 6 and 16 are chosen by the designer (often following classic architectures).
- More output channels allow the network to learn more features, but also increase computation and parameters.

### Why is this a correct order for the training loop?

1. `optimizer.zero_grad()` — Clear old gradients from the last step, so they don't accumulate.
2. `output = net(input)` — Run a forward pass to compute the model's predictions.
3. `loss = criterion(output, target)` — Compute the loss between predictions and true labels.
4. `loss.backward()` — Compute gradients of the loss with respect to all model parameters (backpropagation).
5. `optimizer.step()` — Update the model parameters using the computed gradients.

#### Explanation:
- You must zero the gradients before each backward pass, or gradients will accumulate from previous steps.
- The forward pass and loss calculation must happen before backward, so the computation graph is built and the loss is available.
- Backward computes the gradients needed for the optimizer.
- The optimizer step uses those gradients to update the parameters.
- This order ensures each training step is independent and updates are based only on the current batch.

#### What does `optimizer.step()` do?
- `optimizer.step()` updates the model parameters (weights and biases) using the gradients that were computed during `loss.backward()`.
- For each parameter, it applies the chosen optimization algorithm (e.g., SGD, Adam) to adjust the parameter values in order to minimize the loss.
- In SGD, for example, each parameter is updated by subtracting a fraction of its gradient (scaled by the learning rate and possibly momentum).
- This is the step where learning actually happens: the model's parameters are changed to (hopefully) improve performance on the task.

### What are `net.parameters()` and what does `sub` do?

- `net.parameters()` returns an iterator over all the learnable parameters (weights and biases) of the neural network `net`. These are the tensors that will be updated during training.
    - For example, in a linear layer, parameters include the weight matrix and bias vector.
    - You can loop over them to inspect or update their values.

- `sub` is a PyTorch tensor method that computes the element-wise difference between two tensors and returns a new tensor.
    - Example: `a.sub(b)` returns `a - b` as a new tensor.
    - It does **not** modify `a` in-place.
    - To update `a` in-place, use `sub_`: `a.sub_(b)` changes the values of `a` directly.

#### Summary:
- `net.parameters()` gives all trainable tensors in the model.
- `sub` computes a difference and returns a new tensor; `sub_` updates the tensor in-place.